In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb==0.17.5", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb==0.17.5", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [8]:
%%writefile helm_functional_null_and_specialization.py
"""
HELM functional-null + router-specialization audit.

This script addresses TWO questions that directional effective rank alone cannot:

A) FUNCTIONAL / GEOMETRIC NULL BASELINE
---------------------------------------
Compare trained HELM-7c and Dense-32 against:
  1. the SAME architectures at random initialization, and
  2. a norm-matched random-direction null generated from each trained Gram matrix.

For every requested layer it reports BOTH context-space and residual-space:
  - raw entropy effective rank
  - raw participation rank
  - directional entropy effective rank
  - directional participation rank
  - mean |cosine|
  - norm-matched random-direction null distributions

The norm-matched random-direction null is especially useful:
it preserves the trained per-head energies but randomizes directions.
If trained raw rank ~= norm-matched-null raw rank, then most raw-rank reduction
comes from ENERGY INEQUALITY rather than head-direction alignment.

B) A_h / I_h FUNCTIONAL SPECIALIZATION
---------------------------------------
For HELM-7c elastic head h in layer l:

    A_h = {x : router selected h on x}
    I_h = {x : router did not select h on x}

We use two complementary exact CE tests.

1. Common-coalition test
   Force ALL 32 heads ON in every layer.
   Then remove only (l,h).

   delta_dense(x,l,h) =
       CE(all32 except (l,h)) - CE(all32)

   Compare E[delta_dense | A_h] vs E[delta_dense | I_h].

   This is the cleanest "does the router select h on examples where h is
   actually more useful?" test because both groups are evaluated under the SAME
   all-32 coalition.

2. Policy-marginal test
   Keep HELM's original routed coalition for each example.

   For x in A_h:
       removal_cost = CE(routed with h OFF) - CE(routed)

   For x in I_h:
       addition_benefit = CE(routed) - CE(routed with h ON)

   Positive means "having h helps" in both definitions.

IMPORTANT DATATYPE RULE
-----------------------
All Gram matrices are converted exactly in this order:

    tensor.detach().to(torch.float32).cpu().numpy().astype(np.float64)

This is intentional.

Expected files beside this script:
    model_7c.py
    model_vanilla32.py

or pass explicit --helm-model-file / --dense-model-file paths.

Example (Kaggle TPU):
    python helm_functional_null_and_specialization.py \
        --device xla \
        --helm-model-file model_7c.py \
        --dense-model-file model_vanilla32.py \
        --num-examples 64 \
        --rank-batches 4 \
        --layers 0,5,11 \
        --group-size 8 \
        --ablation-batch-size 2

Outputs:
    helm_functional_audit/
      rank_null_summary.csv
      rank_null_summary.json
      router_activation_summary.csv
      functional_head_summary.csv
      functional_head_example_deltas.csv
      summary.md
      helm_functional_audit_results.zip
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import importlib.util
import json
import math
import os
import random
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required: pip install pyarrow") from exc

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError("huggingface_hub is required: pip install huggingface_hub") from exc


# ---------------------------------------------------------------------------
# Defaults
# ---------------------------------------------------------------------------

HELM_REPO = "JamesResearch1216/HELM_7c"
DENSE_REPO = "JamesResearch1216/HELM_Vanilla_32"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"

DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"


# ---------------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------------

def get_hf_token() -> Optional[str]:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dynamic_import(path: Path, module_name: str):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def strip_state_prefixes(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def parse_layers(text: str) -> List[int]:
    return [int(x.strip()) for x in text.split(",") if x.strip()]


def safe_mean(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(x.mean()) if x.size else float("nan")


def safe_std(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(x.std(ddof=1)) if x.size > 1 else float("nan")


def safe_sem(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    return float(x.std(ddof=1) / math.sqrt(x.size)) if x.size > 1 else float("nan")


def cohens_d_unpaired(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    va = a.var(ddof=1)
    vb = b.var(ddof=1)
    pooled = math.sqrt(
        max(
            0.0,
            ((len(a) - 1) * va + (len(b) - 1) * vb)
            / max(1, len(a) + len(b) - 2),
        )
    )
    if pooled <= 1e-12:
        return float("nan")
    return float((a.mean() - b.mean()) / pooled)


# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------

@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        if self.kind == "xla":
            return torch.autocast(device_type="xla", dtype=torch.bfloat16)
        return contextlib.nullcontext()


def resolve_device(requested: str) -> DeviceContext:
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        else:
            try:
                import torch_xla.core.xla_model as xm
                return DeviceContext(xm.xla_device(), "xla", xm=xm)
            except Exception:
                requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable")
        return DeviceContext(torch.device("cuda"), "cuda")

    if requested == "xla":
        try:
            import torch_xla.core.xla_model as xm
        except Exception as exc:
            raise RuntimeError(
                "XLA requested but torch_xla could not be imported. "
                "Run the normal Kaggle TPU setup/restart cell first."
            ) from exc
        return DeviceContext(xm.xla_device(), "xla", xm=xm)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu")

    raise ValueError(requested)


# ---------------------------------------------------------------------------
# Assets / configs / models
# ---------------------------------------------------------------------------

def download_or_use_checkpoint(
    local_path: Optional[str],
    repo: str,
    filename: str,
    cache_dir: Path,
    token: Optional[str],
    label: str,
) -> Path:
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    print(f"Downloading {label}: {repo}/{filename}")
    p = hf_hub_download(
        repo_id=repo,
        filename=filename,
        repo_type="model",
        token=token,
        local_dir=str(cache_dir / label),
    )
    return Path(p)


def download_training_state(
    repo: str, cache_dir: Path, token: Optional[str]
) -> Optional[Path]:
    try:
        p = hf_hub_download(
            repo_id=repo,
            filename=TRAINING_STATE_FILE,
            repo_type="model",
            token=token,
            local_dir=str(cache_dir / "helm_state"),
        )
        return Path(p)
    except Exception as exc:
        print(f"WARNING: could not download HELM training_state.json: {exc}")
        return None


def load_breakpoints(path: Optional[Path]) -> Optional[List[float]]:
    if path is None or not path.exists():
        return None
    try:
        state = json.loads(path.read_text())
        easiness_dict = state.get("easiness_dict")
        if isinstance(easiness_dict, dict):
            bp = easiness_dict.get("breakpoints")
            if bp:
                return [float(x) for x in bp]
    except Exception as exc:
        print(f"WARNING: failed to parse breakpoints: {exc}")
    return None


def download_validation(
    local_path: Optional[str],
    data_repo: str,
    validation_file: str,
    cache_dir: Path,
    token: Optional[str],
) -> Path:
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    print(f"Downloading validation shard: {data_repo}/{validation_file}")
    p = hf_hub_download(
        repo_id=data_repo,
        filename=validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache_dir / "dataset"),
    )
    return Path(p)


def instantiate_config(module, breakpoints=None):
    kwargs = {}
    # HELM_7c accepts this; Dense-32 may not.
    if breakpoints is not None:
        kwargs["easiness_cdf_breakpoints"] = breakpoints
    try:
        return module.HELMConfig(**kwargs)
    except TypeError:
        return module.HELMConfig()


def load_trained_model(
    module,
    checkpoint: Path,
    dev: DeviceContext,
    breakpoints=None,
    label="model",
):
    config = instantiate_config(module, breakpoints)
    model = module.HELMForMaskedLM(config)

    payload = torch.load(str(checkpoint), map_location="cpu")
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    state = strip_state_prefixes(state)

    incompatible = model.load_state_dict(state, strict=False)
    missing = list(incompatible.missing_keys)
    unexpected = list(incompatible.unexpected_keys)
    if missing or unexpected:
        print(
            f"{label}: {len(missing)} missing / {len(unexpected)} unexpected keys"
        )
        if missing:
            print("  missing:", missing[:10])
        if unexpected:
            print("  unexpected:", unexpected[:10])
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError(
                f"Large state mismatch for {label}. Check the architecture file."
            )

    del payload
    model.to(dev.device)
    model.eval()
    if hasattr(model, "enable_efficient_inference"):
        model.enable_efficient_inference("dense", compile=False)
    return model, config


def make_random_init_model(module, config, dev: DeviceContext):
    # Build a fresh config from the same fields when possible.
    try:
        cfg_dict = config.to_dict()
        fresh_config = module.HELMConfig(**cfg_dict)
    except Exception:
        fresh_config = module.HELMConfig()

    model = module.HELMForMaskedLM(fresh_config)

    # Match the user's trainer behavior: fresh runs explicitly apply _init_weights.
    if hasattr(model, "_init_weights"):
        model.apply(model._init_weights)
    if hasattr(model, "normalize_ngpt_matrices"):
        model.normalize_ngpt_matrices()

    model.to(dev.device)
    model.eval()
    if hasattr(model, "enable_efficient_inference"):
        model.enable_efficient_inference("dense", compile=False)
    return model


# ---------------------------------------------------------------------------
# Deterministic MLM masking / batches
# ---------------------------------------------------------------------------

def deterministic_span_mask(
    ids: torch.Tensor,
    config,
    seed: int,
    probability: float = 0.30,
    span_length: int = 3,
):
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)

    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id),
        int(config.eos_token_id),
        int(config.pad_token_id),
        int(config.mask_token_id),
        int(config.unk_token_id),
    }
    candidate = [i for i, tok in enumerate(ids.tolist()) if int(tok) not in special]
    if not candidate:
        return ids, labels

    target = max(1, int(round(probability * len(candidate))))
    candidate_set = set(candidate)
    perm = torch.randperm(len(candidate), generator=g).tolist()

    chosen = set()
    for pi in perm:
        if len(chosen) >= target:
            break
        start = candidate[pi]
        for pos in range(start, min(start + span_length, ids.numel())):
            if pos in candidate_set:
                chosen.add(pos)
                if len(chosen) >= target:
                    break

    chosen = sorted(chosen)
    if not chosen:
        chosen = [candidate[0]]

    pos = torch.tensor(chosen, dtype=torch.long)
    labels[pos] = ids[pos]

    r = torch.rand(len(pos), generator=g)
    mask_sel = r < 0.80
    random_sel = (r >= 0.80) & (r < 0.90)

    ids[pos[mask_sel]] = int(config.mask_token_id)
    if random_sel.any():
        ids[pos[random_sel]] = torch.randint(
            0,
            int(config.vocab_size),
            (int(random_sel.sum()),),
            generator=g,
        )
    return ids, labels


def prepare_examples(
    validation_path: Path,
    config,
    num_examples: int,
    seq_len: int,
    seed: int,
):
    table = pq.read_table(
        str(validation_path), columns=["input_ids", "easiness_score"]
    )
    n = min(int(num_examples), table.num_rows)
    rng = np.random.default_rng(seed)
    rows = rng.permutation(table.num_rows)[:n]

    input_col = table.column("input_ids")
    easy_col = table.column("easiness_score")

    examples = []
    for i, row_idx in enumerate(rows.tolist()):
        ids = torch.tensor(input_col[row_idx].as_py(), dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            ids = torch.cat(
                [
                    ids,
                    torch.full(
                        (seq_len - ids.numel(),),
                        int(config.pad_token_id),
                        dtype=torch.long,
                    ),
                ]
            )

        masked, labels = deterministic_span_mask(
            ids, config, seed=seed + 100003 * i
        )

        examples.append(
            {
                "input_ids": masked,
                "labels": labels,
                "attention_mask": (masked != int(config.pad_token_id)).long(),
                "easiness_score": torch.tensor(
                    float(easy_col[row_idx].as_py()), dtype=torch.float32
                ),
                "example_id": torch.tensor(i, dtype=torch.long),
            }
        )
    return examples


def make_batches(examples, batch_size: int):
    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    batches = []
    for start in range(0, usable, batch_size):
        chunk = examples[start : start + batch_size]
        batches.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0].keys()
            }
        )
    return examples, batches


def make_subbatches(examples, indices, batch_size: int):
    indices = list(indices)
    usable = (len(indices) // batch_size) * batch_size
    indices = indices[:usable]
    batches = []
    for start in range(0, len(indices), batch_size):
        idx = indices[start : start + batch_size]
        chunk = [examples[i] for i in idx]
        batches.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0].keys()
            }
        )
    return batches


def move_batch(batch, dev: DeviceContext):
    return {k: v.to(dev.device) for k, v in batch.items()}


# ---------------------------------------------------------------------------
# Model-call / CE helpers
# ---------------------------------------------------------------------------

def call_model(model, batch):
    kwargs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
    }
    # Both supplied architectures accept these in the versions used in HELM,
    # but stay defensive.
    try:
        out = model(
            **kwargs,
            current_step=6500,
            easiness_score=None,
        )
    except TypeError:
        out = model(**kwargs)

    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def per_example_ce(logits, labels, chunk_tokens: int = 128):
    B, S, V = logits.shape
    sums = torch.zeros(B, device=logits.device, dtype=torch.float32)
    counts = torch.zeros(B, device=logits.device, dtype=torch.float32)

    for start in range(0, S, chunk_tokens):
        end = min(start + chunk_tokens, S)
        lgt = logits[:, start:end, :].to(torch.float32)
        lab = labels[:, start:end]

        losses = F.cross_entropy(
            lgt.reshape(-1, V),
            lab.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, end - start)

        valid = (lab != -100).to(torch.float32)
        sums += (losses * valid).sum(dim=1)
        counts += valid.sum(dim=1)

    return sums / counts.clamp_min(1.0)


def evaluate_per_example(model, batches, dev: DeviceContext):
    out = {}
    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(model, batch)
                ces = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            ids = batch["example_id"].detach().cpu().tolist()
            vals = ces.detach().to(torch.float32).cpu().tolist()
            for i, v in zip(ids, vals):
                out[int(i)] = float(v)
    return out


# ---------------------------------------------------------------------------
# Router mask override / capture
# ---------------------------------------------------------------------------

class RouterOverride:
    """
    Hook HELM-7c router outputs.

    mode:
      dense              -> all 32 on
      dense_minus        -> all 32 on except one (layer,head)
      forced             -> explicit full [B,H] masks for each layer
    """

    def __init__(
        self,
        model,
        mode: str,
        target_layer: Optional[int] = None,
        target_head: Optional[int] = None,
        forced_masks: Optional[Dict[int, torch.Tensor]] = None,
    ):
        self.model = model
        self.mode = mode
        self.target_layer = target_layer
        self.target_head = target_head
        self.forced_masks = forced_masks or {}
        self.handles = []

    def _hook(self, layer_idx):
        def hook(module, inputs, output):
            B, H, _, _ = output.shape

            if self.mode == "dense":
                return torch.ones_like(output)

            if self.mode == "dense_minus":
                result = torch.ones_like(output)
                if layer_idx == self.target_layer:
                    result[:, self.target_head, :, :] = 0
                return result

            if self.mode == "forced":
                mask = self.forced_masks[layer_idx].to(
                    device=output.device, dtype=output.dtype
                )
                return mask.view(B, H, 1, 1)

            raise ValueError(self.mode)

        return hook

    def __enter__(self):
        for li, block in enumerate(self.model.model.blocks):
            self.handles.append(
                block.mlt_vw_rtr.register_forward_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def capture_router_masks(model, batches, dev: DeviceContext):
    n_layers = len(model.model.blocks)
    masks_by_id = {}
    routed_ce = {}

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(model, batch)
                ces = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            # [B,L,H]
            full_layer_masks = []
            P = int(model.config.num_permanent_heads)
            for li, block in enumerate(model.model.blocks):
                elastic = (
                    block.mlt_vw_rtr.save_hard_mask.detach()
                    .to(torch.float32)
                    .cpu()
                )
                perm = torch.ones(
                    elastic.size(0), P, dtype=torch.float32
                )
                full = torch.cat([perm, elastic], dim=-1)
                full_layer_masks.append(full)
            full_masks = torch.stack(full_layer_masks, dim=1)

            ids = batch["example_id"].detach().cpu().tolist()
            ce_vals = ces.detach().to(torch.float32).cpu().tolist()

            for bi, eid in enumerate(ids):
                masks_by_id[int(eid)] = full_masks[bi].numpy().astype(np.float32)
                routed_ce[int(eid)] = float(ce_vals[bi])

    return masks_by_id, routed_ce


# ---------------------------------------------------------------------------
# Attention geometry
# ---------------------------------------------------------------------------

class AttentionInputCapture:
    def __init__(self, model, layers: Sequence[int]):
        self.model = model
        self.layers = list(layers)
        self.handles = []
        self.data = {}

    def _hook(self, li):
        def hook(module, inputs):
            self.data[li] = (
                inputs[0].detach(),
                inputs[1].detach(),
            )
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(
                    self._hook(li)
                )
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def unmasked_attention_context(model_module, attn, hidden_states, attention_mask):
    cast_linear = model_module.cast_linear
    justnorm = model_module.justnorm

    qkv_proj = cast_linear(hidden_states, attn.qkv)
    B, S, _ = hidden_states.shape

    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)

    q = q.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)

    q = justnorm(q)
    k = justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (
        attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale
    )
    sqk = sqk.view(1, attn.num_attention_heads, 1, attn.d_head).to(q.dtype)
    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (context * vn).sum(dim=-1, keepdim=True) * vn

    return context


def effective_ranks_from_gram(gram: np.ndarray):
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)

    total = vals.sum()
    if total <= 1e-18:
        return {
            "entropy_rank": 0.0,
            "participation_rank": 0.0,
            "eigenvalues": vals,
        }

    p = vals / total
    pp = p[p > 1e-15]
    entropy_rank = float(np.exp(-(pp * np.log(pp)).sum()))
    participation_rank = float(
        (total * total) / (np.square(vals).sum() + 1e-18)
    )
    return {
        "entropy_rank": entropy_rank,
        "participation_rank": participation_rank,
        "eigenvalues": vals,
    }


def directional_gram(raw_gram: np.ndarray):
    diag = np.clip(np.diag(raw_gram), 1e-18, None)
    denom = np.sqrt(np.outer(diag, diag))
    corr = raw_gram / denom
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def mean_abs_offdiag(mat: np.ndarray):
    if mat.shape[0] <= 1:
        return 0.0
    mask = ~np.eye(mat.shape[0], dtype=bool)
    return float(np.abs(mat[mask]).mean())


def add_gram(accum, key, gram_np):
    if key not in accum:
        accum[key] = np.zeros_like(gram_np, dtype=np.float64)
    accum[key] += gram_np


def geometry_for_model(
    label: str,
    model,
    model_module,
    batches,
    dev: DeviceContext,
    layers,
    max_batches: int,
    sample_tokens: int,
    elastic_subset: Optional[Tuple[int, int]] = None,
):
    """
    Returns raw context/residual Gram matrices.

    Conversion order is EXACTLY:
        detach -> float32 -> cpu -> numpy -> float64
    """
    grams = {}
    dims = {}

    with torch.no_grad():
        for cpu_batch in batches[:max_batches]:
            batch = move_batch(cpu_batch, dev)

            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = call_model(model, batch)
                dev.mark_step()

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn

                with dev.autocast():
                    context = unmasked_attention_context(
                        model_module, attn, hidden, attn_mask
                    )
                    S = context.size(2)
                    T = min(int(sample_tokens), S)
                    pos = torch.linspace(
                        0, S - 1, steps=T, device=context.device
                    ).long()
                    c = context.index_select(2, pos)  # [B,H,T,Dh]

                    W = attn.output.weight.to(c.dtype).view(
                        attn.hidden_size,
                        attn.num_attention_heads,
                        attn.d_head,
                    )
                    y = torch.einsum("bhtd,ohd->bhto", c, W)

                    # [H, B*T*D]
                    cflat = (
                        c.permute(1, 0, 2, 3)
                        .contiguous()
                        .view(attn.num_attention_heads, -1)
                    )
                    yflat = (
                        y.permute(1, 0, 2, 3)
                        .contiguous()
                        .view(attn.num_attention_heads, -1)
                    )

                    cgram = cflat @ cflat.T
                    ygram = yflat @ yflat.T

                dev.mark_step()

                # USER-SPECIFIED SAFE CONVERSION ORDER.
                cgram_np = (
                    cgram.detach()
                    .to(torch.float32)
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )
                ygram_np = (
                    ygram.detach()
                    .to(torch.float32)
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )

                add_gram(grams, (li, "all", "context"), cgram_np)
                add_gram(grams, (li, "all", "residual"), ygram_np)
                dims[(li, "all", "context")] = int(cflat.shape[1])
                dims[(li, "all", "residual")] = int(yflat.shape[1])

                if elastic_subset is not None:
                    s0, s1 = elastic_subset
                    add_gram(
                        grams,
                        (li, "elastic", "context"),
                        cgram_np[s0:s1, s0:s1],
                    )
                    add_gram(
                        grams,
                        (li, "elastic", "residual"),
                        ygram_np[s0:s1, s0:s1],
                    )
                    dims[(li, "elastic", "context")] = int(cflat.shape[1])
                    dims[(li, "elastic", "residual")] = int(yflat.shape[1])

                del context, c, y, cflat, yflat, cgram, ygram

    return grams, dims


def norm_matched_random_null(
    gram: np.ndarray,
    vector_dim: int,
    repeats: int,
    seed: int,
    max_sim_dim: int = 65536,
):
    """
    Random directions preserving the trained head energies.

    For very large flattened vectors, use up to max_sim_dim dimensions.
    This is conservative: lowering dimension makes random vectors slightly MORE
    correlated and therefore makes the null directional rank slightly LOWER.
    """
    rng = np.random.default_rng(seed)

    H = gram.shape[0]
    d = max(H + 2, min(int(vector_dim), int(max_sim_dim)))
    energies = np.clip(np.diag(gram), 1e-18, None)

    raw_entropy = []
    raw_pr = []
    dir_entropy = []
    dir_pr = []
    mean_abs_cos = []

    for _ in range(int(repeats)):
        z = rng.normal(size=(H, d)).astype(np.float32)
        z /= np.linalg.norm(z, axis=1, keepdims=True) + 1e-12

        corr = (z @ z.T).astype(np.float64)
        # Same trained per-head energies, random directions.
        scale = np.sqrt(energies)
        raw = corr * np.outer(scale, scale)

        rr = effective_ranks_from_gram(raw)
        dr = effective_ranks_from_gram(corr)

        raw_entropy.append(rr["entropy_rank"])
        raw_pr.append(rr["participation_rank"])
        dir_entropy.append(dr["entropy_rank"])
        dir_pr.append(dr["participation_rank"])
        mean_abs_cos.append(mean_abs_offdiag(corr))

    def stats(x):
        x = np.asarray(x, dtype=np.float64)
        return {
            "mean": float(x.mean()),
            "std": float(x.std(ddof=1)) if len(x) > 1 else 0.0,
            "p05": float(np.quantile(x, 0.05)),
            "p50": float(np.quantile(x, 0.50)),
            "p95": float(np.quantile(x, 0.95)),
        }

    return {
        "simulation_dimension": int(d),
        "raw_entropy": stats(raw_entropy),
        "raw_pr": stats(raw_pr),
        "directional_entropy": stats(dir_entropy),
        "directional_pr": stats(dir_pr),
        "mean_abs_cos": stats(mean_abs_cos),
    }


def summarize_geometry(
    label,
    grams,
    dims,
    null_repeats,
    seed,
):
    rows = []
    detail = {}

    for key, gram in sorted(grams.items()):
        li, subset, space = key

        raw = effective_ranks_from_gram(gram)
        dgram = directional_gram(gram)
        directional = effective_ranks_from_gram(dgram)
        cos = mean_abs_offdiag(dgram)

        null = norm_matched_random_null(
            gram,
            vector_dim=dims[key],
            repeats=null_repeats,
            seed=seed + 1009 * li + (0 if space == "context" else 97),
        )

        row = {
            "model": label,
            "layer": li,
            "subset": subset,
            "space": space,
            "heads": gram.shape[0],
            "raw_entropy_rank": raw["entropy_rank"],
            "raw_entropy_ratio": raw["entropy_rank"] / gram.shape[0],
            "raw_participation_rank": raw["participation_rank"],
            "raw_participation_ratio": raw["participation_rank"] / gram.shape[0],
            "directional_entropy_rank": directional["entropy_rank"],
            "directional_entropy_ratio": directional["entropy_rank"] / gram.shape[0],
            "directional_participation_rank": directional["participation_rank"],
            "directional_participation_ratio": directional["participation_rank"] / gram.shape[0],
            "mean_abs_cos": cos,
            "null_raw_entropy_mean": null["raw_entropy"]["mean"],
            "null_raw_entropy_p05": null["raw_entropy"]["p05"],
            "null_raw_entropy_p95": null["raw_entropy"]["p95"],
            "null_directional_entropy_mean": null["directional_entropy"]["mean"],
            "null_directional_entropy_p05": null["directional_entropy"]["p05"],
            "null_directional_entropy_p95": null["directional_entropy"]["p95"],
            "null_mean_abs_cos_mean": null["mean_abs_cos"]["mean"],
            "null_simulation_dimension": null["simulation_dimension"],
            # Positive: trained directional rank exceeds random-direction null.
            "directional_rank_minus_null_mean": (
                directional["entropy_rank"]
                - null["directional_entropy"]["mean"]
            ),
            # Positive: trained raw rank exceeds what its OWN energy profile would
            # get under random directions.
            "raw_rank_minus_norm_matched_null_mean": (
                raw["entropy_rank"] - null["raw_entropy"]["mean"]
            ),
        }
        rows.append(row)
        detail[f"{label}/L{li}/{subset}/{space}"] = {
            "row": row,
            "null": null,
            "raw_eigenvalues": raw["eigenvalues"].tolist(),
            "directional_eigenvalues": directional["eigenvalues"].tolist(),
        }

    return rows, detail


# ---------------------------------------------------------------------------
# Functional A_h / I_h test
# ---------------------------------------------------------------------------

def forced_masks_for_indices(
    masks_by_id: Dict[int, np.ndarray],
    ids: List[int],
    dev: DeviceContext,
    target_layer: int,
    target_head: int,
    target_value: float,
):
    n_layers = next(iter(masks_by_id.values())).shape[0]
    result = {}

    for li in range(n_layers):
        arr = np.stack([masks_by_id[i][li] for i in ids], axis=0)
        if li == target_layer:
            arr[:, target_head] = float(target_value)
        result[li] = torch.tensor(arr, dtype=torch.float32, device=dev.device)

    return result


def evaluate_dense_minus_head(
    model,
    examples,
    indices,
    batch_size,
    dev,
    target_layer,
    target_head,
    dense_baseline_ce,
):
    deltas = []
    rows = []

    batches = make_subbatches(examples, indices, batch_size)
    with RouterOverride(
        model,
        "dense_minus",
        target_layer=target_layer,
        target_head=target_head,
    ):
        values = evaluate_per_example(model, batches, dev)

    for eid in sorted(values):
        delta = values[eid] - dense_baseline_ce[eid]
        deltas.append(delta)
        rows.append((eid, values[eid], dense_baseline_ce[eid], delta))
    return deltas, rows


def evaluate_policy_toggle(
    model,
    examples,
    indices,
    batch_size,
    dev,
    masks_by_id,
    routed_baseline_ce,
    target_layer,
    target_head,
    target_value,
    definition,
):
    """
    definition:
      active_removal: delta = toggled_ce - routed_ce
      inactive_addition: delta = routed_ce - toggled_ce

    Positive always means "having the head helps".
    """
    all_rows = []
    all_deltas = []

    batches = make_subbatches(examples, indices, batch_size)

    for cpu_batch in batches:
        ids = [int(x) for x in cpu_batch["example_id"].tolist()]
        forced = forced_masks_for_indices(
            masks_by_id,
            ids,
            dev,
            target_layer,
            target_head,
            target_value,
        )
        with RouterOverride(model, "forced", forced_masks=forced):
            vals = evaluate_per_example(model, [cpu_batch], dev)

        for eid in ids:
            toggled = vals[eid]
            routed = routed_baseline_ce[eid]
            if definition == "active_removal":
                delta = toggled - routed
            elif definition == "inactive_addition":
                delta = routed - toggled
            else:
                raise ValueError(definition)
            all_deltas.append(delta)
            all_rows.append((eid, toggled, routed, delta))

    return all_deltas, all_rows


def functional_specialization_analysis(
    model,
    examples,
    batches,
    dev,
    masks_by_id,
    routed_ce,
    layers,
    group_size,
    ablation_batch_size,
    seed,
    run_policy_marginal=True,
):
    P = int(model.config.num_permanent_heads)
    H = int(model.config.num_attention_heads)

    # Cache the common all-32 CE ONCE.
    with RouterOverride(model, "dense"):
        dense_ce = evaluate_per_example(model, batches, dev)

    rng = np.random.default_rng(seed)
    summary_rows = []
    example_rows = []

    all_ids = sorted(masks_by_id.keys())

    for li in layers:
        for h in range(P, H):
            active_ids = [
                i for i in all_ids if masks_by_id[i][li, h] > 0.5
            ]
            inactive_ids = [
                i for i in all_ids if masks_by_id[i][li, h] <= 0.5
            ]

            activation_frequency = len(active_ids) / max(1, len(all_ids))

            rng.shuffle(active_ids)
            rng.shuffle(inactive_ids)

            nA = min(int(group_size), len(active_ids))
            nI = min(int(group_size), len(inactive_ids))

            # Keep static complete mini-batches.
            nA = (nA // ablation_batch_size) * ablation_batch_size
            nI = (nI // ablation_batch_size) * ablation_batch_size

            active_use = active_ids[:nA]
            inactive_use = inactive_ids[:nI]

            dense_A, dense_A_rows = ([], [])
            dense_I, dense_I_rows = ([], [])
            pol_A, pol_A_rows = ([], [])
            pol_I, pol_I_rows = ([], [])

            if active_use:
                dense_A, dense_A_rows = evaluate_dense_minus_head(
                    model,
                    examples,
                    active_use,
                    ablation_batch_size,
                    dev,
                    li,
                    h,
                    dense_ce,
                )
            if inactive_use:
                dense_I, dense_I_rows = evaluate_dense_minus_head(
                    model,
                    examples,
                    inactive_use,
                    ablation_batch_size,
                    dev,
                    li,
                    h,
                    dense_ce,
                )

            if run_policy_marginal and active_use:
                pol_A, pol_A_rows = evaluate_policy_toggle(
                    model,
                    examples,
                    active_use,
                    ablation_batch_size,
                    dev,
                    masks_by_id,
                    routed_ce,
                    li,
                    h,
                    target_value=0.0,
                    definition="active_removal",
                )

            if run_policy_marginal and inactive_use:
                pol_I, pol_I_rows = evaluate_policy_toggle(
                    model,
                    examples,
                    inactive_use,
                    ablation_batch_size,
                    dev,
                    masks_by_id,
                    routed_ce,
                    li,
                    h,
                    target_value=1.0,
                    definition="inactive_addition",
                )

            dense_gap = (
                safe_mean(dense_A) - safe_mean(dense_I)
                if dense_A and dense_I
                else float("nan")
            )
            policy_gap = (
                safe_mean(pol_A) - safe_mean(pol_I)
                if pol_A and pol_I
                else float("nan")
            )

            summary_rows.append(
                {
                    "layer": li,
                    "head": h,
                    "elastic_head": h - P,
                    "activation_frequency": activation_frequency,
                    "active_available": len(active_ids),
                    "inactive_available": len(inactive_ids),
                    "active_used": len(active_use),
                    "inactive_used": len(inactive_use),
                    "dense_ablation_active_mean": safe_mean(dense_A),
                    "dense_ablation_active_sem": safe_sem(dense_A),
                    "dense_ablation_inactive_mean": safe_mean(dense_I),
                    "dense_ablation_inactive_sem": safe_sem(dense_I),
                    "dense_specialization_gap_active_minus_inactive": dense_gap,
                    "dense_specialization_cohens_d": cohens_d_unpaired(
                        dense_A, dense_I
                    ),
                    "policy_active_removal_cost_mean": safe_mean(pol_A),
                    "policy_active_removal_cost_sem": safe_sem(pol_A),
                    "policy_inactive_addition_benefit_mean": safe_mean(pol_I),
                    "policy_inactive_addition_benefit_sem": safe_sem(pol_I),
                    "policy_specialization_gap_active_minus_inactive": policy_gap,
                    "policy_specialization_cohens_d": cohens_d_unpaired(
                        pol_A, pol_I
                    ),
                }
            )

            for group_name, rows in (
                ("A_active_dense_ablation", dense_A_rows),
                ("I_inactive_dense_ablation", dense_I_rows),
            ):
                for eid, toggled, baseline, delta in rows:
                    example_rows.append(
                        {
                            "layer": li,
                            "head": h,
                            "elastic_head": h - P,
                            "group": group_name,
                            "example_id": eid,
                            "baseline_ce": baseline,
                            "toggled_ce": toggled,
                            "positive_utility_delta": delta,
                        }
                    )

            for group_name, rows in (
                ("A_active_policy_removal", pol_A_rows),
                ("I_inactive_policy_addition", pol_I_rows),
            ):
                for eid, toggled, baseline, delta in rows:
                    example_rows.append(
                        {
                            "layer": li,
                            "head": h,
                            "elastic_head": h - P,
                            "group": group_name,
                            "example_id": eid,
                            "baseline_ce": baseline,
                            "toggled_ce": toggled,
                            "positive_utility_delta": delta,
                        }
                    )

            print(
                f"L{li:02d} h{h:02d} freq={activation_frequency:.3f} | "
                f"dense gap={dense_gap:+.5f} | policy gap={policy_gap:+.5f}"
            )

    return summary_rows, example_rows, dense_ce


# ---------------------------------------------------------------------------
# CSV / summary helpers
# ---------------------------------------------------------------------------

def write_csv(path: Path, rows: List[dict]):
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def aggregate_functional(summary_rows):
    out = {}
    for li in sorted(set(r["layer"] for r in summary_rows)):
        rr = [r for r in summary_rows if r["layer"] == li]
        out[li] = {
            "mean_activation_frequency": safe_mean(
                [r["activation_frequency"] for r in rr]
            ),
            "mean_dense_specialization_gap": safe_mean(
                [
                    r["dense_specialization_gap_active_minus_inactive"]
                    for r in rr
                ]
            ),
            "fraction_dense_gap_positive": safe_mean(
                [
                    float(
                        r["dense_specialization_gap_active_minus_inactive"] > 0
                    )
                    for r in rr
                    if np.isfinite(
                        r["dense_specialization_gap_active_minus_inactive"]
                    )
                ]
            ),
            "mean_policy_specialization_gap": safe_mean(
                [
                    r["policy_specialization_gap_active_minus_inactive"]
                    for r in rr
                ]
            ),
            "fraction_policy_gap_positive": safe_mean(
                [
                    float(
                        r["policy_specialization_gap_active_minus_inactive"] > 0
                    )
                    for r in rr
                    if np.isfinite(
                        r["policy_specialization_gap_active_minus_inactive"]
                    )
                ]
            ),
        }
    return out


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--device", default="auto", choices=["auto", "cuda", "xla", "cpu"])

    parser.add_argument("--helm-model-file", default="model_7c.py")
    parser.add_argument("--dense-model-file", default="model_vanilla32.py")

    parser.add_argument("--helm-repo", default=HELM_REPO)
    parser.add_argument("--dense-repo", default=DENSE_REPO)
    parser.add_argument("--checkpoint", default=CHECKPOINT_FILE)
    parser.add_argument("--helm-checkpoint-path", default=None)
    parser.add_argument("--dense-checkpoint-path", default=None)

    parser.add_argument("--data-repo", default=DATA_REPO)
    parser.add_argument("--validation-file", default=VALIDATION_FILE)
    parser.add_argument("--validation-path", default=None)

    parser.add_argument("--num-examples", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--seq-len", type=int, default=1024)

    parser.add_argument("--layers", default="0,5,11")
    parser.add_argument("--rank-batches", type=int, default=4)
    parser.add_argument("--sample-tokens", type=int, default=32)
    parser.add_argument("--null-repeats", type=int, default=50)

    parser.add_argument("--group-size", type=int, default=8)
    parser.add_argument("--ablation-batch-size", type=int, default=2)
    parser.add_argument("--skip-policy-marginal", action="store_true")

    parser.add_argument("--seed", type=int, default=1216)
    parser.add_argument("--output-dir", default="helm_functional_audit")
    parser.add_argument("--cache-dir", default=".helm_functional_cache")

    args = parser.parse_args()
    seed_everything(args.seed)

    output_dir = Path(args.output_dir)
    cache_dir = Path(args.cache_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    layers = parse_layers(args.layers)
    dev = resolve_device(args.device)
    token = get_hf_token()

    helm_module = dynamic_import(Path(args.helm_model_file), "helm7c_arch")
    dense_module = dynamic_import(Path(args.dense_model_file), "dense32_arch")

    helm_ckpt = download_or_use_checkpoint(
        args.helm_checkpoint_path,
        args.helm_repo,
        args.checkpoint,
        cache_dir,
        token,
        "helm7c",
    )
    dense_ckpt = download_or_use_checkpoint(
        args.dense_checkpoint_path,
        args.dense_repo,
        args.checkpoint,
        cache_dir,
        token,
        "dense32",
    )
    training_state = download_training_state(
        args.helm_repo, cache_dir, token
    )
    breakpoints = load_breakpoints(training_state)

    validation_path = download_validation(
        args.validation_path,
        args.data_repo,
        args.validation_file,
        cache_dir,
        token,
    )

    print("\n=== Loading trained models ===")
    helm, helm_config = load_trained_model(
        helm_module, helm_ckpt, dev, breakpoints, "HELM-7c trained"
    )
    dense, dense_config = load_trained_model(
        dense_module, dense_ckpt, dev, None, "Dense-32 trained"
    )

    print("\n=== Building random-initialization null models ===")
    helm_random = make_random_init_model(
        helm_module, helm_config, dev
    )
    dense_random = make_random_init_model(
        dense_module, dense_config, dev
    )

    examples = prepare_examples(
        validation_path,
        helm_config,
        args.num_examples,
        args.seq_len,
        args.seed,
    )
    examples, batches = make_batches(examples, args.batch_size)
    print(
        f"Prepared {len(examples)} examples in {len(batches)} batches "
        f"(B={args.batch_size}, S={args.seq_len})"
    )

    # ------------------------------------------------------------------
    # 1. Router masks + baseline CE
    # ------------------------------------------------------------------
    print("\n=== Capturing HELM router decisions ===")
    masks_by_id, routed_ce = capture_router_masks(
        helm, batches, dev
    )

    activation_rows = []
    P = int(helm.config.num_permanent_heads)
    H = int(helm.config.num_attention_heads)
    for li in range(len(helm.model.blocks)):
        for h in range(P, H):
            freq = safe_mean(
                [masks_by_id[i][li, h] for i in sorted(masks_by_id)]
            )
            activation_rows.append(
                {
                    "layer": li,
                    "head": h,
                    "elastic_head": h - P,
                    "activation_frequency": freq,
                }
            )
    write_csv(output_dir / "router_activation_summary.csv", activation_rows)

    # ------------------------------------------------------------------
    # 2. Functional / geometric null baseline
    # ------------------------------------------------------------------
    print("\n=== Rank/null geometry: HELM-7c trained ===")
    h_g, h_d = geometry_for_model(
        "HELM-7c trained",
        helm,
        helm_module,
        batches,
        dev,
        layers,
        args.rank_batches,
        args.sample_tokens,
        elastic_subset=(P, H),
    )

    print("\n=== Rank/null geometry: HELM-7c random init ===")
    hr_g, hr_d = geometry_for_model(
        "HELM-7c random-init",
        helm_random,
        helm_module,
        batches,
        dev,
        layers,
        args.rank_batches,
        args.sample_tokens,
        elastic_subset=(P, H),
    )

    print("\n=== Rank/null geometry: Dense-32 trained ===")
    d_g, d_d = geometry_for_model(
        "Dense-32 trained",
        dense,
        dense_module,
        batches,
        dev,
        layers,
        args.rank_batches,
        args.sample_tokens,
        elastic_subset=None,
    )

    print("\n=== Rank/null geometry: Dense-32 random init ===")
    dr_g, dr_d = geometry_for_model(
        "Dense-32 random-init",
        dense_random,
        dense_module,
        batches,
        dev,
        layers,
        args.rank_batches,
        args.sample_tokens,
        elastic_subset=None,
    )

    rank_rows = []
    rank_detail = {}

    for label, g, d in (
        ("HELM-7c trained", h_g, h_d),
        ("HELM-7c random-init", hr_g, hr_d),
        ("Dense-32 trained", d_g, d_d),
        ("Dense-32 random-init", dr_g, dr_d),
    ):
        rows, detail = summarize_geometry(
            label, g, d, args.null_repeats, args.seed
        )
        rank_rows.extend(rows)
        rank_detail.update(detail)

    write_csv(output_dir / "rank_null_summary.csv", rank_rows)
    (output_dir / "rank_null_summary.json").write_text(
        json.dumps(rank_detail, indent=2)
    )

    # ------------------------------------------------------------------
    # 3. A_h / I_h functional specialization
    # ------------------------------------------------------------------
    print("\n=== A_h / I_h exact functional specialization ===")
    functional_rows, example_rows, dense_ce = functional_specialization_analysis(
        helm,
        examples,
        batches,
        dev,
        masks_by_id,
        routed_ce,
        layers,
        args.group_size,
        args.ablation_batch_size,
        args.seed,
        run_policy_marginal=(not args.skip_policy_marginal),
    )

    write_csv(output_dir / "functional_head_summary.csv", functional_rows)
    write_csv(
        output_dir / "functional_head_example_deltas.csv", example_rows
    )

    functional_agg = aggregate_functional(functional_rows)

    # ------------------------------------------------------------------
    # 4. Human-readable summary
    # ------------------------------------------------------------------
    lines = []
    lines.append("# HELM functional null + specialization audit")
    lines.append("")
    lines.append("## Interpretation rules")
    lines.append("")
    lines.append(
        "- Directional rank near the random-init / random-direction null is **not evidence of specialization**."
    )
    lines.append(
        "- `raw_rank_minus_norm_matched_null_mean ≈ 0` means the trained raw-rank deficit is explained mostly by its per-head energy profile rather than extra directional alignment."
    )
    lines.append(
        "- Positive `dense_specialization_gap_active_minus_inactive` means the router tends to select head h on examples where h has larger exact utility under the same all-32 coalition."
    )
    lines.append(
        "- Positive `policy_specialization_gap_active_minus_inactive` means h is more useful in its actually selected routed coalitions than it would be when inserted into rejected coalitions."
    )
    lines.append("")

    lines.append("## Rank/null snapshots")
    lines.append("")
    lines.append(
        "| model | layer | subset | space | raw eRank | dir eRank | random-null dir eRank | mean|cos| |"
    )
    lines.append("|---|---:|---|---|---:|---:|---:|---:|")
    for r in rank_rows:
        lines.append(
            f"| {r['model']} | {r['layer']} | {r['subset']} | {r['space']} | "
            f"{r['raw_entropy_rank']:.2f}/{r['heads']} | "
            f"{r['directional_entropy_rank']:.2f}/{r['heads']} | "
            f"{r['null_directional_entropy_mean']:.2f}/{r['heads']} | "
            f"{r['mean_abs_cos']:.4f} |"
        )

    lines.append("")
    lines.append("## Functional specialization aggregates")
    lines.append("")
    lines.append(
        "| layer | mean active freq | mean dense A-I gap | frac dense gap > 0 | mean policy A-I gap | frac policy gap > 0 |"
    )
    lines.append("|---:|---:|---:|---:|---:|---:|")
    for li, r in functional_agg.items():
        lines.append(
            f"| {li} | {r['mean_activation_frequency']:.3f} | "
            f"{r['mean_dense_specialization_gap']:+.5f} | "
            f"{r['fraction_dense_gap_positive']:.3f} | "
            f"{r['mean_policy_specialization_gap']:+.5f} | "
            f"{r['fraction_policy_gap_positive']:.3f} |"
        )

    lines.append("")
    lines.append("## Key causal question")
    lines.append("")
    lines.append(
        "If the router learned genuine head/example specialization, then for a meaningful fraction of elastic heads we should see:"
    )
    lines.append("")
    lines.append("`E[ΔCE_h | A_h] > E[ΔCE_h | I_h]`")
    lines.append("")
    lines.append(
        "under the common all-32 coalition, not merely high directional rank."
    )

    (output_dir / "summary.md").write_text("\n".join(lines))

    # Zip outputs.
    # IMPORTANT: create the archive OUTSIDE output_dir so the ZIP
    # cannot recursively include itself.
    zip_base = output_dir.parent / "helm_functional_audit_results"
    
    # Remove an old archive if one already exists.
    old_zip = zip_base.with_suffix(".zip")
    if old_zip.exists():
        old_zip.unlink()
    
    zip_path = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=output_dir,
    )
    
    print(f"\nDone. Results directory: {output_dir}")
    print(f"Results ZIP: {zip_path}")


if __name__ == "__main__":
    main()

Overwriting helm_functional_null_and_specialization.py


In [3]:
%%writefile model_7c.py

##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm_7c"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # HELM_7c Router
        num_router_latents = 4,
        num_permanent_heads = 8,
        head_target_min = 8,
        head_target_center = 16,
        head_target_max = 32,
        easiness_cdf_breakpoints = None,
        count_loss_lambda = 0.5,
        router_grad_clip = 0.05,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # HELM_7c Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.count_loss_lambda = count_loss_lambda
        self.router_grad_clip = router_grad_clip
        self.jitter_noise = jitter_noise

        elastic = num_attention_heads - num_permanent_heads
        if num_permanent_heads != head_target_min:
            raise ValueError(
                "HELM_7c uses permanent heads as the structural minimum; "
                "num_permanent_heads must equal head_target_min."
            )
        if elastic <= 0:
            raise ValueError("HELM_7c requires at least one elastic head")
        if not (head_target_min <= head_target_center <= head_target_max <= num_attention_heads):
            raise ValueError("Invalid HELM_7c head targets")

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# HELM_7c multi-latent router
class HELMMultiViewRouter(nn.Module):
    """Minimal sequence-level elastic router for HELM_7c.

    There are 8 permanent heads and 24 elastic candidates. The elastic router uses
    ordinary learned logits z_h(x). Forward routing is hard: z_h > 0. The same hard
    mask is wrapped in a sigmoid STE so CE and the count loss can train the router.

    Easiness labels are training-time supervision only. They are converted to a
    desired total head count in [8, 32]. At inference no easiness value is needed.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads

        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )
        self.l_i_weights = nn.Parameter(torch.ones(config.num_router_latents))

        # IMPORTANT: unlike Phase 13, q_up_proj is NOT normalized. Magnitude is allowed
        # to carry information. We monitor its norms instead of pre-emptively constraining it.
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # Last-forward telemetry.
        self.save_router_logits = None
        self.save_sigmoid_scores = None
        self.save_hard_mask = None
        self.save_total_head_count = None
        self.save_target_total_head_count = None
        self.save_count_error = None
        self.save_count_loss = None

    def _easiness_to_target(self, easiness_score):
        """Map easiness label -> integer target total heads in [8, 32].

        Easiness is converted to a CDF quantile q so the target depends on relative
        difficulty rather than the raw label's dataset-specific numeric scale:
          q=0   (hardest) -> 32 total heads
          q=0.5 (median)  -> 16 total heads
          q=1   (easiest) -> 8 total heads
        """
        batch = easiness_score.numel()
        device = easiness_score.device
        e = easiness_score.to(torch.float32).reshape(batch).clamp(0.0, 1.0)

        bp = self.config.easiness_cdf_breakpoints
        if bp is not None and len(bp) >= 2:
            breaks = torch.as_tensor(bp, device=device, dtype=torch.float32)
            n_intervals = breaks.numel() - 1
            pos = torch.searchsorted(breaks, e, right=True).clamp(1, n_intervals)
            lo = breaks[pos - 1]
            hi = breaks[pos]
            frac = (e - lo) / (hi - lo + 1e-8)
            q = ((pos - 1).to(torch.float32) + frac) / float(n_intervals)
            q = q.clamp(0.0, 1.0)
        else:
            # Safe fallback if no breakpoint table was supplied.
            q = e

        h_min = float(self.config.head_target_min)
        h_ctr = float(self.config.head_target_center)
        h_max = float(self.config.head_target_max)

        hard_half = q < 0.5
        hard_target = h_ctr + (h_max - h_ctr) * ((0.5 - q) / 0.5)
        easy_target = h_ctr + (h_min - h_ctr) * ((q - 0.5) / 0.5)
        target_total = torch.where(hard_half, hard_target, easy_target)

        # Actual executed counts are integer, so make an exactly attainable target.
        return target_total.round().clamp(h_min, h_max)

    def forward(self, hidden_states, easiness_score=None):
        # ----- Existing HELM multi-latent sequence summary -----
        q_down = justnorm(self.q_down_proj.weight, dim=1).to(hidden_states.dtype)
        scanner = F.linear(hidden_states, q_down)                       # [B,S,R]
        scanner_weights = F.softmax(self.scale * scanner, dim=1)       # [B,S,R]
        latents = torch.bmm(scanner_weights.transpose(1, 2), hidden_states)  # [B,R,D]

        latent_weights = F.softmax(self.l_i_weights, dim=0)
        pooled = (latents * latent_weights.view(1, -1, 1)).sum(dim=1)  # [B,D]

        # ----- Minimal learned router -----
        router_logits = cast_linear(pooled, self.q_up_proj)                 # [B,E]
        sigmoid_scores = torch.sigmoid(router_logits)
        hard_mask = (router_logits > 0).to(router_logits.dtype)

        # Forward = exact 0/1 hard mask. Backward = sigmoid derivative.
        ste_mask = hard_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        actual_elastic_count = hard_mask.sum(dim=-1)
        actual_total_count = actual_elastic_count + float(self.config.num_permanent_heads)

        # ----- Easiness-supervised ACTUAL hard-count loss -----
        if easiness_score is not None:
            target_total_count = self._easiness_to_target(easiness_score)
            target_elastic_count = target_total_count - float(self.config.num_permanent_heads)

            # ste_mask has the hard count as its forward value but keeps a sigmoid
            # backward path. This avoids the old sum(sigmoid) soft-count loophole.
            differentiable_elastic_count = ste_mask.float().sum(dim=-1)
            count_error = differentiable_elastic_count - target_elastic_count.float()
            denom = float(self.num_elastic_candidates)
            count_loss = (
                float(self.config.count_loss_lambda)
                * (count_error / denom).square().mean()
            )
        else:
            if self.training:
                raise ValueError("HELM_7c training requires easiness_score")
            target_total_count = torch.full_like(actual_total_count, -1.0)
            count_error = torch.zeros_like(actual_total_count)
            count_loss = router_logits.new_zeros(())

        # ----- Telemetry -----
        self.save_router_logits = router_logits.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()
        self.save_hard_mask = hard_mask.detach()
        self.save_total_head_count = actual_total_count.detach()
        self.save_target_total_head_count = target_total_count.detach()
        self.save_count_error = (actual_total_count - target_total_count).detach()
        self.count_loss = count_loss
        self.save_count_loss = count_loss.detach()

        router_mask = ste_mask.view(ste_mask.size(0), -1, 1, 1)
        if self.config.num_permanent_heads > 0:
            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )
            router_mask = torch.cat((permanent, router_mask), dim=1)

        return router_mask


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.d_head if config.d_head is not None else (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head   
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))  # was: self.hidden_size

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,      # was: config.hidden_size
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, _ = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention + HELMMLP
class HELMBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, easiness_score):
        router_mask = self.mlt_vw_rtr(hidden_states, easiness_score)
        count_loss = self.mlt_vw_rtr.count_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, count_loss


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, easiness_score=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_count_loss = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, count_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, count_loss = block(hidden_states, attention_mask, easiness_score)
            total_count_loss = total_count_loss + count_loss

        # Count supervision is per-layer; average so lambda is independent of depth.
        total_count_loss = total_count_loss / float(len(self.blocks))
        return hidden_states, total_count_loss


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is intentionally EXCLUDED: HELM_7c allows router magnitude.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr
            logits = router.save_router_logits.float().cpu()
            sigmoid = router.save_sigmoid_scores.float().cpu()
            hard = router.save_hard_mask.float().cpu()
            actual = router.save_total_head_count.float().cpu()
            target = router.save_target_total_head_count.float().cpu()
            error = router.save_count_error.float().cpu()
            q_up_norms = router.q_up_proj.weight.detach().float().norm(dim=1).cpu()

            telemetry[f"layer_{i}_router_logits"] = logits
            telemetry[f"layer_{i}_sigmoid_scores"] = sigmoid
            telemetry[f"layer_{i}_hard_mask"] = hard
            telemetry[f"layer_{i}_elastic_head_ratio"] = hard.mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = actual.mean().item()
            telemetry[f"layer_{i}_target_head_count_mean"] = target.mean().item()
            telemetry[f"layer_{i}_count_error_mean"] = error.mean().item()
            telemetry[f"layer_{i}_count_error_mae"] = error.abs().mean().item()
            telemetry[f"layer_{i}_count_loss"] = router.save_count_loss.float().item()
            telemetry[f"layer_{i}_router_weight_norms"] = q_up_norms
            telemetry[f"layer_{i}_router_weight_norm_mean"] = q_up_norms.mean().item()
            telemetry[f"layer_{i}_router_weight_norm_std"] = q_up_norms.std(unbiased=False).item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # current_step is accepted only for backward compatibility with older callers.
        features, total_count_loss = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            easiness_score=easiness_score,
        )

        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_count_loss



Writing model_7c.py


In [4]:
%%writefile model_vanilla32.py
"""HELM Dense-32 baseline.

Clean dense control for HELM_7c:
- 12 layers
- d_model = 1024
- 32 attention heads
- d_head = 64
- total attention width = 2048
- all 32 heads active for every example in every layer
- no router, no permanent/elastic split, no easiness supervision,
  no count loss, and no router-specific jitter.

The nGPT attention/MLP mechanics are intentionally kept aligned with HELM_7c.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None

from transformers import PretrainedConfig, PreTrainedModel


def justnorm(x, dim=-1, eps=1e-12):
    return x / (x.norm(p=2, dim=dim, keepdim=True) + eps)


def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x, w, b)


class HELMConfig(PretrainedConfig):
    model_type = "helm_dense32"

    def __init__(
        self,
        # General model hyperparameters
        hidden_size=1024,
        sqrt_hidden_size=32,
        max_position_embeddings=4096,
        initializer_range=0.03125,
        num_hidden_layers=12,
        num_attention_heads=32,
        d_head=64,
        rope_theta=160000,
        intermediate_size=2816,
        norm_eps=1e-12,
        hidden_act="swiglu",
        swiglu_s_init=1.0,
        base_lr=3e-4,
        min_lr=3e-5,
        weight_decay=0.0,
        bias=False,
        use_ckpt=False,

        # Tokenization / MLM metadata
        tokenizer_path="answerdotai/ModernBERT-base",
        vocab_size=50368,
        bos_token_id=50281,
        eos_token_id=50282,
        pad_token_id=50283,
        mask_token_id=50284,
        unk_token_id=50285,
        mlm_probability=0.3,
        mlm_use_span_masking=True,
        mlm_span_length=3,

        # nGPT attention / FFN hyperparameters
        ngpt_sqk_init_value=1.0,
        ngpt_sqk_init_scale=0.03125,
        use_exclusive_attention=True,
        ngpt_alpha_value_attn=0.05,
        ngpt_alpha_scale_attn=0.03125,
        ngpt_alpha_value_mlp=0.05,
        ngpt_alpha_scale_mlp=0.03125,
        ngpt_suv_value=1.0,
        ngpt_suv_scale=1.0,
        ngpt_sz_init_value=1.0,
        ngpt_sz_init_scale=0.03125,
        **kwargs,
    ):
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale

        if self.num_attention_heads != 32:
            raise ValueError(
                f"Dense-32 control requires num_attention_heads=32, got {self.num_attention_heads}."
            )
        if self.d_head != 64:
            raise ValueError(f"Dense-32 control requires d_head=64, got {self.d_head}.")
        if self.num_attention_heads * self.d_head != 2048:
            raise ValueError("Dense-32 attention width must be exactly 2048.")

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id,
        )

    def forward(self, input_ids):
        return justnorm(self.word_embeddings(input_ids))


class RotaryEmbeddings(nn.Module):
    def __init__(self, dim, max_position_embeddings, rope_theta=160000):
        super().__init__()
        inv_freq = 1.0 / (
            rope_theta ** (torch.arange(0, dim, 2).float() / dim)
        )
        t = torch.arange(max_position_embeddings, dtype=inv_freq.dtype)
        freqs = torch.outer(t, inv_freq)
        freqs = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x):
        seq_len = x.shape[-2]
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)
        return (x * x_cos) + (self.rotate_half(x) * x_sin)


class HELMSelfAttention(nn.Module):
    """Dense 32-head attention. Every head is always active."""

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.d_head = config.d_head
        self.total_head_dim = self.num_attention_heads * self.d_head
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        # 1024 -> Q/K/V, each 2048 wide.
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias=config.bias,
        )

        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta,
        )

        self.sqk = nn.Parameter(
            self.ngpt_sqk_init_scale * torch.ones(self.total_head_dim)
        )

        # 2048 -> 1024, matching HELM_7c.
        self.output = nn.Linear(
            self.total_head_dim,
            config.hidden_size,
            bias=config.bias,
        )

    def forward(self, hidden_states, attention_mask):
        batch_size, seq_len, _ = hidden_states.shape

        qkv_proj = cast_linear(hidden_states, self.qkv)
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        q = q.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        k = k.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        v = v.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)

        # Keep HELM/nGPT mechanics unchanged.
        q = justnorm(q)
        k = justnorm(k)
        q = self.RoPE(q)
        k = self.RoPE(k)

        sqk = self.sqk * (
            self.ngpt_sqk_init_value / self.ngpt_sqk_init_scale
        )
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)
        q = sqk.to(q.dtype) * q
        k = sqk.to(k.dtype) * k

        context_layer = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask.to(q.dtype),
            scale=math.sqrt(self.d_head),
        )

        if self.config.use_exclusive_attention:
            vn = F.normalize(v, dim=-1)
            context_layer = (
                context_layer
                - (context_layer * vn).sum(dim=-1, keepdim=True) * vn
            )

        context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
        context_reshaped = context_reshaped.view(batch_size, seq_len, self.total_head_dim)
        return cast_linear(context_reshaped, self.output)


class HELMMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        self.attn_alpha = nn.Parameter(
            self.ngpt_alpha_scale_attn * torch.ones(self.hidden_size)
        )
        self.mlp_alpha = nn.Parameter(
            self.ngpt_alpha_scale_mlp * torch.ones(self.hidden_size)
        )

        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias=config.bias,
        )
        self.suv = nn.Parameter(
            self.ngpt_suv_scale * torch.ones(2 * self.intermediate_size)
        )
        self.silu = nn.SiLU()
        self.mlp_proj = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias,
        )

    def forward(self, hidden_states, hidden_states_attention):
        # nGPT attention residual update.
        a_norm = justnorm(hidden_states)
        b_norm = justnorm(hidden_states_attention)
        lr = self.attn_alpha * (
            self.ngpt_alpha_value_attn / self.ngpt_alpha_scale_attn
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        hidden_states_opt1 = justnorm(a_norm + lr * (b_norm - a_norm))

        # nGPT SwiGLU FFN.
        uv_pre = cast_linear(hidden_states_opt1, self.mlp_exp)
        suv = self.suv * (
            self.ngpt_suv_value / self.ngpt_suv_scale
        ) * (self.hidden_size ** 0.5)
        uv_post_suv = suv.to(uv_pre.dtype) * uv_pre
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)
        x_mlp = u * self.silu(v)
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # nGPT MLP residual update.
        a_norm = justnorm(hidden_states_opt1)
        b_norm = justnorm(h_mlp)
        lr = self.mlp_alpha * (
            self.ngpt_alpha_value_mlp / self.ngpt_alpha_scale_mlp
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        return justnorm(a_norm + lr * (b_norm - a_norm))


class HELMBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask):
        attn_output = self.attn(hidden_states, attention_mask)
        return self.mlp(hidden_states, attn_output)


class HELMModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, input_ids, attention_mask):
        # Additive SDPA mask: [B,S] -> [B,1,1,S].
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(
            attention_mask == 0, float("-inf")
        )
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)

        for block in self.blocks:
            if self.use_ckpt and self.training:
                ckpt_fn = (
                    _xla_checkpoint
                    if (
                        _xla_checkpoint is not None
                        and hidden_states.device.type == "xla"
                    )
                    else torch.utils.checkpoint.checkpoint
                )
                hidden_states = ckpt_fn(
                    block,
                    hidden_states,
                    attention_mask,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states = block(hidden_states, attention_mask)

        return hidden_states


class HELMForMaskedLM(PreTrainedModel):
    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=config.bias,
        )
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # Every matrix here existed in the dense-16 baseline as well.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}
        for i, block in enumerate(self.model.blocks):
            attn = block.attn
            mlp = block.mlp

            sqk = attn.sqk.detach().float().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk.mean().item()
            telemetry[f"layer_{i}_sqk_std"] = sqk.std(unbiased=False).item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk

            attn_alpha = mlp.attn_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha

            mlp_alpha = mlp.mlp_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha

            suv = mlp.suv.detach().float().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv.std(unbiased=False).item()
            telemetry[f"layer_{i}_suv_hist"] = suv

        sz = self.sz.detach().float().cpu()
        telemetry["lm_head_sz_mean"] = sz.mean().item()
        telemetry["lm_head_sz_std"] = sz.std(unbiased=False).item()
        telemetry["lm_head_sz_hist"] = sz
        return telemetry

    def forward(
        self,
        input_ids,
        attention_mask,
        current_step=None,
        easiness_score=None,
    ):
        # current_step/easiness_score are accepted only so old generic callers do
        # not break. They have ZERO effect on this dense model.
        features = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sz = self.sz * (
            self.ngpt_sz_init_value / self.ngpt_sz_init_scale
        )
        unscaled_logits = cast_linear(features, self.classifier)
        return sz.to(unscaled_logits.dtype) * unscaled_logits

Writing model_vanilla32.py


In [9]:
!python helm_functional_null_and_specialization.py \
    --device xla \
    --helm-model-file model_7c.py \
    --dense-model-file model_vanilla32.py \
    --num-examples 64 \
    --batch-size 2 \
    --layers 0,5,11 \
    --rank-batches 4 \
    --group-size 8 \
    --ablation-batch-size 2

/kaggle/working/helm_functional_null_and_specialization.py:279: DeprecationWarning: Use torch_xla.device instead
  return DeviceContext(xm.xla_device(), "xla", xm=xm)
E0000 00:00:1786561627.180064    1971 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238

=== Loading trained models ===

=== Building random-initialization null models ===
Prepared 64 examples in 32 batches (B=2, S=1024)

=== Capturing HELM router decisions ===
/kaggle/working/helm_functional_null_and_specialization.py:244: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()

=== Rank/null geometry: HELM-7c trained ===

=== Rank/null geometry: HELM-7c random init ===

=== Rank/null geometry: Dense-32 trained ===

=== Rank/null geometry: Dense-32 random init ===

=== A_h / I_h exact functional specializatio